# RAG Práctica – Implementación Comparativa de Dos Pipelines
Este notebook implementa dos configuraciones completas de RAG utilizando como corpus el archivo de texto: "Law-for-Computer-Scientists".  
Seguiremos la estructura:
1. Carga del corpus
2. Limpieza y chunking
3. Embeddings Modelo A
4. Búsqueda y métricas para Modelo A
5. Generación con LLM A
6. Embeddings Modelo B
7. Búsqueda y métricas Modelo B
8. Generación con LLM B
9. Comparación final de precisión y coherencia


In [3]:
print("="*80)
print("STEP 1: LOAD EMBEDDING MODELS A & B")
print("="*80)

from sentence_transformers import SentenceTransformer
import numpy as np

embedding_models = {
    "A": SentenceTransformer("intfloat/e5-base"),
    "B": SentenceTransformer("BAAI/bge-small-en-v1.5")
}

print("Embedding models loaded successfully.")


STEP 1: LOAD EMBEDDING MODELS A & B
Embedding models loaded successfully.


El texto contiene:
- Encabezados repetidos
- Saltos de línea duplicados
- Numeración de páginas

Realizamos limpieza básica antes de chunkear.


In [4]:
print("="*80)
print("STEP 2: LOAD TEXT & CREATE CHUNKS")
print("="*80)

with open("Law-for-Computer-Scientists.txt", "r", encoding="utf-8") as f:
    texto_limpio = f.read()

def crear_chunks(texto, size=800, overlap=300):
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fin = inicio + size
        chunks.append(texto[inicio:fin])
        inicio += size - overlap
    return chunks

chunks = crear_chunks(texto_limpio)

print(f"Chunks created: {len(chunks)}")


STEP 2: LOAD TEXT & CREATE CHUNKS
Chunks created: 1487


Dividimos en segmentos de 600 caracteres con un solapamiento de 150, para preservar contexto en temas legales largos.


In [5]:
print("="*80)
print("STEP 3: GENERATE EMBEDDINGS")
print("="*80)

embeddings = {}

for key, model in embedding_models.items():
    print(f"Generating embeddings for model {key}...")
    emb = model.encode(chunks, show_progress_bar=True)
    embeddings[key] = emb

print("Embeddings ready.")


STEP 3: GENERATE EMBEDDINGS
Generating embeddings for model A...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Generating embeddings for model B...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Embeddings ready.


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(impl_key, pregunta, topk=3):
    q = embedding_models[impl_key].encode([pregunta])
    sims = cosine_similarity(q, embeddings[impl_key])[0]
    idx = np.argsort(sims)[::-1][:topk]
    return idx, sims[idx]


In [7]:
def create_rag_prompt(question, retrieved_chunks):
    context = ""
    for i, ch in enumerate(retrieved_chunks, 1):
        context += f"[Fragment {i}]:\n{ch}\n\n"

    return f"""
You are an assistant restricted to answer ONLY with information found in the fragments.

STRICT RULES:
1. Use ONLY information explicitly stated in the fragments.
2. When using information from a fragment, cite it like: (Fragment X).
3. If the answer is NOT found in the fragments, respond EXACTLY:
   "I cannot find that information in the provided fragments."
4. Do NOT add explanations, reasoning steps, summaries, or comments.
5. Do NOT generate questions.
6. Your output must contain ONLY the final answer.

FRAGMENTS:
{context}

QUESTION: {question}

ANSWER:
"""


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

generators = {
    "A": "microsoft/Phi-3-mini-4k-instruct",       # 2.3 GB
    "B": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"      # 2.2 GB
}

generator_pipes = {}

def load_generator(repo):
    print(f"\nLoading generator: {repo}")

    tokenizer = AutoTokenizer.from_pretrained(repo)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        repo,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=200,
        temperature=0.0,   # sigue estrictamente instrucciones
        do_sample=False
    )

print("Loading both generator models once...\n")
generator_pipes["A"] = load_generator(generators["A"])
generator_pipes["B"] = load_generator(generators["B"])
print("\nBoth generators loaded successfully.")


Loading both generator models once...


Loading generator: microsoft/Phi-3-mini-4k-instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0



Both generators loaded successfully.


In [9]:
def run_rag(impl_key, question, k=3):
    idxs, sims = retrieve(impl_key, question, k)
    selected_chunks = [chunks[i] for i in idxs]

    # CONTEXTO
    print("==============================================")
    print("        CONTEXTO UTILIZADO POR EL RAG         ")
    print("==============================================\n")
    for pos, (chunk_i, sim) in enumerate(zip(idxs, sims)):
        print(f"--- Chunk {chunk_i}  |  similitud = {sim:.4f} ---\n")
        print(chunks[chunk_i])
        print("\n----------------------------------------------\n")

    # PROMPT
    prompt = create_rag_prompt(question, selected_chunks)
    pipe = generator_pipes[impl_key]

    # GENERACIÓN
    raw = pipe(prompt)[0]["generated_text"]

    # Cortar donde empieza la respuesta
    if "ANSWER:" in raw:
        raw = raw.split("ANSWER:", 1)[1].strip()

    # Cortar cualquier continuación
    output = raw.split("\n\n")[0].strip()

    print("============= RESPUESTA DEL MODELO =============")
    print(output)
    print("=================================================\n")

    return {
        "impl": impl_key,
        "chunks": idxs.tolist(),
        "similarities": sims.tolist(),
        "answer": output
    }


In [14]:
# Preguntas
questions = [
    "What are the three cumulative legal conditions required for an invention to qualify as a patent?"
]

results = {"A": [], "B": []}

# Loop por cada pregunta
for q in questions:

    print("\n" + "="*80)
    print(f"QUESTION: {q}")
    print("="*80)

    print("\n>>> IMPLEMENTATION A (E5 embeddings + Phi-3 generator)\n")
    res_A = run_rag("A", q)
    results["A"].append(res_A)

    print("\n>>> IMPLEMENTATION B (BGE embeddings + Gemma generator)\n")
    res_B = run_rag("B", q)
    results["B"].append(res_B)

print("\n===== RAG experiment finished successfully =====")



QUESTION: What are the three cumulative legal conditions required for an invention to qualify as a patent?

>>> IMPLEMENTATION A (E5 embeddings + Phi-3 generator)

        CONTEXTO UTILIZADO POR EL RAG         

--- Chunk 920  |  similitud = 0.8646 ---

ights in
‘design’, in ‘databases’, and in ‘software’ (note that in the United States one can
obtain a patent in software, whereas in Europe that is only possible if the soft-
ware is part of an ‘invention’ that has a material component).
Sections 7.3 and 7.4 will further explore copyright.


7.2.2 Patents

Like copyright, a patent is limited in time and jurisdiction. The time period
during which a patent is valid again depends on national jurisdiction.

 To qualify as a patent the intellectual good must be:

      1. an invention,
      2. that is novel, and
      3. has an industrial application.


In other words, to qualify as patentable, these three legal conditions must
apply (they are cumulative). The legal effect of being granted

In [12]:
# Preguntas
questions = [
    "How do constitutional law and international public law differ in their relevance to cybercrime?",
]

results = {"A": [], "B": []}

# Loop por cada pregunta
for q in questions:

    print("\n" + "="*80)
    print(f"QUESTION: {q}")
    print("="*80)

    print("\n>>> IMPLEMENTATION A (E5 embeddings + Phi-3 generator)\n")
    res_A = run_rag("A", q)
    results["A"].append(res_A)

    print("\n>>> IMPLEMENTATION B (BGE embeddings + Gemma generator)\n")
    res_B = run_rag("B", q)
    results["B"].append(res_B)

print("\n===== RAG experiment finished successfully =====")



QUESTION: How do constitutional law and international public law differ in their relevance to cybercrime?

>>> IMPLEMENTATION A (E5 embeddings + Phi-3 generator)

        CONTEXTO UTILIZADO POR EL RAG         

--- Chunk 786  |  similitud = 0.9032 ---

ion. Because cybercrime does
not stop at national borders, states are collaborating at the international
and supranational level to come to terms with the transnational nature of
cybercrime.
168 Cybercrime

6.2 Cybercrime and Public Law

As discussed above, public law consists of constitutional law, international
public law, and administrative law. Constitutional law is relevant for cyber-
crime to the extent that it determines the right to a fair trial, the criminal
law legality principle, and the right to privacy (that is often at stake when
states create and apply investigatory competences to combat cybercrime).
International public law is relevant for cybercrime because the need to act
across territorial borders has resulted in con